In [1]:
import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.cuda.is_available(): True
torch.cuda.device_count(): 1
GPU: Tesla T4


In [2]:
%pip install -q transformers datasets accelerate sentencepiece scikit-learn

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

ZIP_PATTERNS = {
    "fp2021": "feedback-prize-2021",
    "fpe": "feedback-prize-effectiveness",
    "ell": "feedback-prize-english-language-learning",
    "patent": "us-patent-phrase-to-phrase-matching",
}

found_zips = {}

for key, name_part in ZIP_PATTERNS.items():
    matches = list(DRIVE_ROOT.rglob(f"*{name_part}*.zip"))
    if matches:
        found_zips[key] = matches[0]
    else:
        found_zips[key] = None

print("Found zip files:")
for k, v in found_zips.items():
    print(k, "->", v)

Found zip files:
fp2021 -> /content/drive/MyDrive/feedback-prize-2021.zip
fpe -> /content/drive/MyDrive/feedback-prize-effectiveness.zip
ell -> /content/drive/MyDrive/feedback-prize-english-language-learning.zip
patent -> /content/drive/MyDrive/us-patent-phrase-to-phrase-matching.zip


In [5]:
import zipfile
from pathlib import Path

EXTRACT_ROOT = Path("/content/eduai_datasets")
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

for key, zip_path in found_zips.items():
    if zip_path is None:
        print(f"Missing zip for {key}")
        continue

    out_dir = EXTRACT_ROOT / key
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    print(f"Extracted {key} -> {out_dir}")

Extracted fp2021 -> /content/eduai_datasets/fp2021
Extracted fpe -> /content/eduai_datasets/fpe
Extracted ell -> /content/eduai_datasets/ell
Extracted patent -> /content/eduai_datasets/patent


In [6]:
from pathlib import Path

def find_first(root: Path, pattern: str):
    matches = list(root.rglob(pattern))
    return matches[0] if matches else None

fp2021_train_csv = find_first(EXTRACT_ROOT / "fp2021", "train.csv")
fpe_train_csv = find_first(EXTRACT_ROOT / "fpe", "train.csv")
ell_train_csv = find_first(EXTRACT_ROOT / "ell", "train.csv")

print("fp2021_train_csv:", fp2021_train_csv)
print("fpe_train_csv   :", fpe_train_csv)
print("ell_train_csv   :", ell_train_csv)

fp2021_train_csv: /content/eduai_datasets/fp2021/train.csv
fpe_train_csv   : /content/eduai_datasets/fpe/train.csv
ell_train_csv   : /content/eduai_datasets/ell/train.csv


In [7]:
import pandas as pd

fp2021_train = pd.read_csv(fp2021_train_csv)
fpe_train = pd.read_csv(fpe_train_csv)
ell_train = pd.read_csv(ell_train_csv)

print("FP2021 shape:", fp2021_train.shape)
print("FPE shape   :", fpe_train.shape)
print("ELL shape   :", ell_train.shape)

FP2021 shape: (144293, 8)
FPE shape   : (36765, 5)
ELL shape   : (3911, 8)


In [8]:
import re
import random
import numpy as np
import torch
import warnings

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def clean_text(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text: str) -> int:
    return len(clean_text(text).split())

def keep_text(text: str, min_words: int = 6, max_words: int = 300) -> bool:
    n = word_count(text)
    return min_words <= n <= max_words

torch.cuda.is_available(): True
torch.cuda.device_count(): 1
GPU: Tesla T4


In [9]:
disc3_labels = ["structure", "evidence", "argument"]

FP_MAP = {
    "Evidence": "evidence",
    "Claim": "argument",
    "Counterclaim": "argument",
    "Rebuttal": "argument",
    "Position": "argument",
    "Lead": "structure",
    "Concluding Statement": "structure",
}

EFFECTIVENESS_WEIGHT = {
    "Effective": 1.20,
    "Adequate": 1.00,
    "Ineffective": 0.75,
}

fp2021_mapped = fp2021_train.copy()
fp2021_mapped["label"] = fp2021_mapped["discourse_type"].map(FP_MAP)
fp2021_mapped = fp2021_mapped[fp2021_mapped["label"].isin(disc3_labels)].copy()

fp2021_samples = pd.DataFrame({
    "text": fp2021_mapped["discourse_text"].map(clean_text),
    "label": fp2021_mapped["label"],
    "source": "fp2021",
    "weight": 1.0
})
fp2021_samples = fp2021_samples[fp2021_samples["text"].map(keep_text)].copy()

fpe_mapped = fpe_train.copy()
fpe_mapped["label"] = fpe_mapped["discourse_type"].map(FP_MAP)
fpe_mapped = fpe_mapped[fpe_mapped["label"].isin(disc3_labels)].copy()

fpe_samples = pd.DataFrame({
    "text": fpe_mapped["discourse_text"].map(clean_text),
    "label": fpe_mapped["label"],
    "source": "fpe",
    "weight": fpe_mapped["discourse_effectiveness"].map(EFFECTIVENESS_WEIGHT).fillna(1.0)
})
fpe_samples = fpe_samples[fpe_samples["text"].map(keep_text)].copy()

disc3_df = pd.concat([fp2021_samples, fpe_samples], ignore_index=True)

disc3_df = (
    disc3_df
    .groupby(["text", "label"], as_index=False)
    .agg(
        weight=("weight", "mean"),
        source=("source", lambda s: "|".join(sorted(set(s))))
    )
)

print("3-class discourse dataset:", disc3_df.shape)
print(disc3_df["label"].value_counts())
display(disc3_df.sample(min(10, len(disc3_df)), random_state=SEED))

3-class discourse dataset: (138533, 4)
label
argument     70381
evidence     45439
structure    22713
Name: count, dtype: int64


,text,label,weight,source
43059,"If you get multiple opinions, you will have th...",argument,1.0,fp2021
109873,While the current schooling system does serve ...,structure,1.1,fp2021|fpe
136921,while others genuinely have a busy schedule on...,argument,1.0,fp2021
21324,Every student in Generic_City has heard of a P...,argument,1.0,fp2021
118685,by introducing their mind to different points ...,argument,1.1,fp2021|fpe
51260,It avoids that problem because candidates can ...,evidence,1.0,fp2021
121461,i am not in favor of keeping the Electoral Col...,argument,1.0,fp2021
134693,they still need to be driven by a human to get...,evidence,1.0,fp2021|fpe
34276,I hope that each principal at each middle scho...,structure,1.0,fp2021|fpe
119687,end humans from making mistakes on the road,argument,1.0,fp2021


In [10]:
from sklearn.model_selection import train_test_split

disc3_train, disc3_temp = train_test_split(
    disc3_df,
    test_size=0.20,
    stratify=disc3_df["label"],
    random_state=SEED
)

disc3_val, disc3_test = train_test_split(
    disc3_temp,
    test_size=0.50,
    stratify=disc3_temp["label"],
    random_state=SEED
)

print("Train:")
print(disc3_train["label"].value_counts())

print("\nVal:")
print(disc3_val["label"].value_counts())

print("\nTest:")
print(disc3_test["label"].value_counts())

Train:
label
argument     56305
evidence     36351
structure    18170
Name: count, dtype: int64

Val:
label
argument     7038
evidence     4544
structure    2271
Name: count, dtype: int64

Test:
label
argument     7038
evidence     4544
structure    2272
Name: count, dtype: int64


In [11]:
def rebalance_exact(df, target_per_class=9000, seed=42):
    parts = []
    for label, group in df.groupby("label"):
        if len(group) >= target_per_class:
            group = group.sample(target_per_class, random_state=seed)
        else:
            group = group.sample(target_per_class, replace=True, random_state=seed)
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

disc3_train_bal = rebalance_exact(disc3_train, target_per_class=9000, seed=SEED)

print(disc3_train_bal.shape)
print(disc3_train_bal["label"].value_counts())

(27000, 4)
label
argument     9000
structure    9000
evidence     9000
Name: count, dtype: int64


In [12]:
disc3_label2id = {label: i for i, label in enumerate(disc3_labels)}
disc3_id2label = {i: label for label, i in disc3_label2id.items()}

disc3_label2id, disc3_id2label

({'structure': 0, 'evidence': 1, 'argument': 2},
 {0: 'structure', 1: 'evidence', 2: 'argument'})

In [13]:
from datasets import Dataset

train_hf = disc3_train_bal.copy()
val_hf = disc3_val.copy()
test_hf = disc3_test.copy()

train_hf["labels"] = train_hf["label"].map(disc3_label2id)
val_hf["labels"] = val_hf["label"].map(disc3_label2id)
test_hf["labels"] = test_hf["label"].map(disc3_label2id)

train_ds = Dataset.from_pandas(train_hf[["text", "labels", "weight"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_hf[["text", "labels", "weight"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_hf[["text", "labels", "weight"]], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['text', 'labels', 'weight'],
    num_rows: 27000
})
Dataset({
    features: ['text', 'labels', 'weight'],
    num_rows: 13853
})
Dataset({
    features: ['text', 'labels', 'weight'],
    num_rows: 13854
})


In [14]:
from transformers import AutoTokenizer

MODEL_NAME = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=192
    )

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

print(train_ds.column_names)
print(val_ds.column_names)
print(test_ds.column_names)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/13853 [00:00<?, ? examples/s]

Map:   0%|          | 0/13854 [00:00<?, ? examples/s]

['labels', 'weight', 'input_ids', 'attention_mask']
['labels', 'weight', 'input_ids', 'attention_mask']
['labels', 'weight', 'input_ids', 'attention_mask']


In [15]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

class SampleWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        sample_weight = inputs.pop("weight")
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none", label_smoothing=0.02)
        losses = loss_fct(logits, labels)

        sample_weight = sample_weight.to(losses.device).float()
        loss = (losses * sample_weight).mean()

        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(disc3_labels),
    id2label=disc3_id2label,
    label2id=disc3_label2id,
    ignore_mismatched_sizes=True
)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
training_args = TrainingArguments(
    output_dir="/content/discourse3_distilroberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_weighted_f1",
    greater_is_better=True,

    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.08,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,

    fp16=True,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [18]:
trainer = SampleWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [19]:
val_pred_output = trainer.predict(val_ds)
val_preds = np.argmax(val_pred_output.predictions, axis=1)
val_true = np.array(val_ds["labels"])

print("Validation Accuracy   :", accuracy_score(val_true, val_preds))
print("Validation Macro F1   :", f1_score(val_true, val_preds, average="macro"))
print("Validation Weighted F1:", f1_score(val_true, val_preds, average="weighted"))
print("\nValidation report:\n")
print(classification_report(
    val_true,
    val_preds,
    target_names=disc3_labels,
    digits=4
))

Validation Accuracy   : 0.15917129863567459
Validation Macro F1   : 0.09355294736722476
Validation Weighted F1: 0.04722857588140009

Validation report:

              precision    recall  f1-score   support

   structure     0.1618    0.9652    0.2771      2271
    evidence     0.0000    0.0000    0.0000      4544
    argument     0.0428    0.0018    0.0035      7038

    accuracy                         0.1592     13853
   macro avg     0.0682    0.3224    0.0936     13853
weighted avg     0.0482    0.1592    0.0472     13853



In [20]:
test_pred_output = trainer.predict(test_ds)
test_preds = np.argmax(test_pred_output.predictions, axis=1)
test_true = np.array(test_ds["labels"])

print("Test Accuracy   :", accuracy_score(test_true, test_preds))
print("Test Macro F1   :", f1_score(test_true, test_preds, average="macro"))
print("Test Weighted F1:", f1_score(test_true, test_preds, average="weighted"))
print("\nTest report:\n")
print(classification_report(
    test_true,
    test_preds,
    target_names=disc3_labels,
    digits=4
))

Test Accuracy   : 0.1603147105529089
Test Macro F1   : 0.09453221156355261
Test Weighted F1: 0.04819253343669658

Test report:

              precision    recall  f1-score   support

   structure     0.1627    0.9696    0.2787      2272
    evidence     0.0000    0.0000    0.0000      4544
    argument     0.0568    0.0026    0.0049      7038

    accuracy                         0.1603     13854
   macro avg     0.0732    0.3241    0.0945     13854
weighted avg     0.0555    0.1603    0.0482     13854



In [21]:
from pathlib import Path
import pandas as pd
import re
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def clean_text(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text: str) -> int:
    return len(clean_text(text).split())

def keep_text(text: str, min_words: int = 6, max_words: int = 300) -> bool:
    n = word_count(text)
    return min_words <= n <= max_words

disc3_labels = ["structure", "evidence", "argument"]

FP_MAP = {
    "Evidence": "evidence",
    "Claim": "argument",
    "Counterclaim": "argument",
    "Rebuttal": "argument",
    "Position": "argument",
    "Lead": "structure",
    "Concluding Statement": "structure",
}

EFFECTIVENESS_WEIGHT = {
    "Effective": 1.20,
    "Adequate": 1.00,
    "Ineffective": 0.75,
}

fp2021_mapped = fp2021_train.copy()
fp2021_mapped["label"] = fp2021_mapped["discourse_type"].map(FP_MAP)
fp2021_mapped = fp2021_mapped[fp2021_mapped["label"].isin(disc3_labels)].copy()

fp2021_samples = pd.DataFrame({
    "text": fp2021_mapped["discourse_text"].map(clean_text),
    "label": fp2021_mapped["label"],
})
fp2021_samples = fp2021_samples[fp2021_samples["text"].map(keep_text)].copy()

fpe_mapped = fpe_train.copy()
fpe_mapped["label"] = fpe_mapped["discourse_type"].map(FP_MAP)
fpe_mapped = fpe_mapped[fpe_mapped["label"].isin(disc3_labels)].copy()

fpe_samples = pd.DataFrame({
    "text": fpe_mapped["discourse_text"].map(clean_text),
    "label": fpe_mapped["label"],
})
fpe_samples = fpe_samples[fpe_samples["text"].map(keep_text)].copy()

disc3_df = pd.concat([fp2021_samples, fpe_samples], ignore_index=True)
disc3_df = disc3_df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print(disc3_df.shape)
print(disc3_df["label"].value_counts())

(138533, 2)
label
argument     70381
evidence     45439
structure    22713
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split

disc3_train, disc3_temp = train_test_split(
    disc3_df,
    test_size=0.20,
    stratify=disc3_df["label"],
    random_state=SEED
)

disc3_val, disc3_test = train_test_split(
    disc3_temp,
    test_size=0.50,
    stratify=disc3_temp["label"],
    random_state=SEED
)

print("Train:")
print(disc3_train["label"].value_counts())
print("\nVal:")
print(disc3_val["label"].value_counts())
print("\nTest:")
print(disc3_test["label"].value_counts())

Train:
label
argument     56305
evidence     36351
structure    18170
Name: count, dtype: int64

Val:
label
argument     7038
evidence     4544
structure    2271
Name: count, dtype: int64

Test:
label
argument     7038
evidence     4544
structure    2272
Name: count, dtype: int64


In [23]:
def rebalance_exact(df, target_per_class=9000, seed=42):
    parts = []
    for label, group in df.groupby("label"):
        if len(group) >= target_per_class:
            group = group.sample(target_per_class, random_state=seed)
        else:
            group = group.sample(target_per_class, replace=True, random_state=seed)
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

disc3_train_bal = rebalance_exact(disc3_train, target_per_class=9000, seed=SEED)

print(disc3_train_bal.shape)
print(disc3_train_bal["label"].value_counts())

(27000, 2)
label
argument     9000
structure    9000
evidence     9000
Name: count, dtype: int64


In [24]:
from datasets import Dataset

disc3_label2id = {label: i for i, label in enumerate(disc3_labels)}
disc3_id2label = {i: label for label, i in disc3_label2id.items()}

train_hf = disc3_train_bal.copy()
val_hf = disc3_val.copy()
test_hf = disc3_test.copy()

train_hf["labels"] = train_hf["label"].map(disc3_label2id)
val_hf["labels"] = val_hf["label"].map(disc3_label2id)
test_hf["labels"] = test_hf["label"].map(disc3_label2id)

train_ds = Dataset.from_pandas(train_hf[["text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_hf[["text", "labels"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_hf[["text", "labels"]], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['text', 'labels'],
    num_rows: 27000
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 13853
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 13854
})


In [25]:
from transformers import AutoTokenizer

MODEL_NAME = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=192
    )

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

print(train_ds.column_names)
print(val_ds.column_names)
print(test_ds.column_names)

Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/13853 [00:00<?, ? examples/s]

Map:   0%|          | 0/13854 [00:00<?, ? examples/s]

['labels', 'input_ids', 'attention_mask']
['labels', 'input_ids', 'attention_mask']
['labels', 'input_ids', 'attention_mask']


In [26]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(disc3_labels),
    id2label=disc3_id2label,
    label2id=disc3_label2id,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
training_args = TrainingArguments(
    output_dir="/content/discourse3_distilroberta_clean",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_weighted_f1",
    greater_is_better=True,

    num_train_epochs=3,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,

    fp16=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.606017,0.541730,0.805313,0.781158,0.810187
2,0.412864,0.461713,0.819678,0.796235,0.822207
3,0.298514,0.522011,0.817729,0.795392,0.821120


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=5064, training_loss=0.4391316900509417, metrics={'train_runtime': 490.1838, 'train_samples_per_second': 165.244, 'train_steps_per_second': 10.331, 'total_flos': 3311845447760784.0, 'train_loss': 0.4391316900509417, 'epoch': 3.0})

In [29]:
import re
import random
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def clean_text(text: str) -> str:
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text: str) -> int:
    return len(clean_text(text).split())

In [30]:
FP_MAP = {
    "Evidence": "evidence",
    "Claim": "argument",
    "Counterclaim": "argument",
    "Rebuttal": "argument",
    "Position": "argument",
    "Lead": "structure",
    "Concluding Statement": "structure",
}

disc3_labels = ["structure", "evidence", "argument"]

fp = fp2021_train.copy()
fp["label"] = fp["discourse_type"].map(FP_MAP)
fp = fp[fp["label"].isin(disc3_labels)].copy()

print(fp.shape)
print(fp["label"].value_counts())

(144293, 9)
label
argument     75781
evidence     45702
structure    22810
Name: count, dtype: int64


In [32]:
from pathlib import Path

EXTRACT_ROOT = Path("/content/eduai_datasets")

fp2021_essay_dir = None
for p in (EXTRACT_ROOT / "fp2021").rglob("*"):
    if p.is_dir() and p.name.lower() == "train":
        txts = list(p.glob("*.txt"))
        if txts:
            fp2021_essay_dir = p
            break

print("fp2021_essay_dir:", fp2021_essay_dir)
print("txt count:", len(list(fp2021_essay_dir.glob('*.txt'))) if fp2021_essay_dir else 0)

fp2021_essay_dir: /content/eduai_datasets/fp2021/train
txt count: 15594


In [33]:
essay_texts = {}
for txt_file in fp2021_essay_dir.glob("*.txt"):
    essay_texts[txt_file.stem] = txt_file.read_text(encoding="utf-8", errors="ignore")

print("Loaded essays:", len(essay_texts))

Loaded essays: 15594


In [34]:
def safe_text(x):
    return clean_text(x) if pd.notna(x) else ""

rows = []

for essay_id, group in fp.groupby("id", sort=False):
    group = group.sort_values("discourse_start").reset_index(drop=True)
    essay_text = essay_texts.get(str(essay_id), "")
    essay_len = max(len(essay_text), 1)

    for i, row in group.iterrows():
        current_text = safe_text(row["discourse_text"])
        prev_text = safe_text(group.iloc[i - 1]["discourse_text"]) if i > 0 else ""
        next_text = safe_text(group.iloc[i + 1]["discourse_text"]) if i < len(group) - 1 else ""

        start = int(row["discourse_start"]) if pd.notna(row["discourse_start"]) else 0
        end = int(row["discourse_end"]) if pd.notna(row["discourse_end"]) else start

        span_len_chars = max(end - start, 0)
        span_wc = word_count(current_text)

        start_ratio = start / essay_len
        end_ratio = end / essay_len
        mid_ratio = ((start + end) / 2) / essay_len

        position_bucket = (
            "intro" if mid_ratio < 0.20 else
            "early" if mid_ratio < 0.40 else
            "middle" if mid_ratio < 0.60 else
            "late" if mid_ratio < 0.80 else
            "conclusion"
        )

        is_first = int(i == 0)
        is_last = int(i == len(group) - 1)

        rows.append({
            "essay_id": row["id"],
            "label": row["label"],
            "text": current_text,
            "prev_text": prev_text,
            "next_text": next_text,
            "context_text": f"{prev_text} [SEP] {current_text} [SEP] {next_text}",
            "start_ratio": start_ratio,
            "end_ratio": end_ratio,
            "mid_ratio": mid_ratio,
            "span_len_chars": span_len_chars,
            "span_len_words": span_wc,
            "position_bucket": position_bucket,
            "is_first_discourse": is_first,
            "is_last_discourse": is_last,
        })

disc_ctx_df = pd.DataFrame(rows)

print(disc_ctx_df.shape)
print(disc_ctx_df["label"].value_counts())
display(disc_ctx_df.head())

(144293, 14)
label
argument     75781
evidence     45702
structure    22810
Name: count, dtype: int64


,essay_id,label,text,prev_text,next_text,context_text,start_ratio,end_ratio,mid_ratio,span_len_chars,span_len_words,position_bucket,is_first_discourse,is_last_discourse
0,423A1CA112E2,structure,Modern humans today are always on their phone....,,They are some really bad consequences when stu...,[SEP] Modern humans today are always on their...,0.003941,0.112808,0.058374,221,44,intro,1,0
1,423A1CA112E2,argument,They are some really bad consequences when stu...,Modern humans today are always on their phone....,Some certain areas in the United States ban ph...,Modern humans today are always on their phone....,0.113300,0.153695,0.133498,82,15,intro,0,0
2,423A1CA112E2,evidence,Some certain areas in the United States ban ph...,They are some really bad consequences when stu...,"When people have phones, they know about certa...",They are some really bad consequences when stu...,0.154187,0.197537,0.175862,88,16,intro,0,0
3,423A1CA112E2,evidence,"When people have phones, they know about certa...",Some certain areas in the United States ban ph...,Driving is one of the way how to get around. P...,Some certain areas in the United States ban ph...,0.198030,0.373399,0.285714,356,63,early,0,0
4,423A1CA112E2,argument,Driving is one of the way how to get around. P...,"When people have phones, they know about certa...",That's why there's a thing that's called no te...,"When people have phones, they know about certa...",0.373892,0.436453,0.405172,127,24,middle,0,0


In [35]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(gss1.split(disc_ctx_df, groups=disc_ctx_df["essay_id"]))

train_df = disc_ctx_df.iloc[train_idx].reset_index(drop=True)
temp_df = disc_ctx_df.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["essay_id"]))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print("Train:", train_df.shape)
print(train_df["label"].value_counts())
print("\nVal:", val_df.shape)
print(val_df["label"].value_counts())
print("\nTest:", test_df.shape)
print(test_df["label"].value_counts())

Train: (115436, 14)
label
argument     60670
evidence     36508
structure    18258
Name: count, dtype: int64

Val: (14494, 14)
label
argument     7552
evidence     4656
structure    2286
Name: count, dtype: int64

Test: (14363, 14)
label
argument     7559
evidence     4538
structure    2266
Name: count, dtype: int64


In [36]:
def rebalance_focus(df, seed=42):
    targets = {
        "structure": 16000,
        "evidence": 12000,
        "argument": 12000,
    }

    parts = []
    for label, group in df.groupby("label"):
        target = targets[label]
        if len(group) >= target:
            group = group.sample(target, random_state=seed)
        else:
            group = group.sample(target, replace=True, random_state=seed)
        parts.append(group)

    out = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

train_bal = rebalance_focus(train_df, seed=SEED)

print(train_bal.shape)
print(train_bal["label"].value_counts())

(40000, 14)
label
structure    16000
evidence     12000
argument     12000
Name: count, dtype: int64


In [37]:
text_features = ["text", "prev_text", "next_text", "context_text"]
numeric_features = [
    "start_ratio", "end_ratio", "mid_ratio",
    "span_len_chars", "span_len_words",
    "is_first_discourse", "is_last_discourse"
]
categorical_features = ["position_bucket"]

preprocessor = ColumnTransformer(
    transformers=[
        ("text_main", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            max_features=120000
        ), "text"),
        
        ("text_prev", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            max_features=40000
        ), "prev_text"),
        
        ("text_next", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            max_features=40000
        ), "next_text"),
        
        ("text_context_char", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True,
            max_features=60000
        ), "context_text"),
        
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ],
    remainder="drop"
)

In [38]:
class_weight_focus = {
    "structure": 1.6,
    "evidence": 1.0,
    "argument": 0.95,
}

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=5000,
        solver="saga",
        n_jobs=-1,
        C=1.5,
        class_weight=class_weight_focus,
        random_state=SEED
    ))
])

model.fit(train_bal, train_bal["label"])

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('text_main',
                                                  TfidfVectorizer(max_df=0.95,
                                                                  max_features=120000,
                                                                  min_df=2,
                                                                  ngram_range=(1,
                                                                               3),
                                                                  strip_accents='unicode',
                                                                  sublinear_tf=True),
                                                  'text'),
                                                 ('text_prev',
                                                  TfidfVectorizer(max_df=0.98,
                                                                  max_features=40000,
                                                                  min_df=2,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  strip_accents='unicode',
                                                                  sublinear_tf=True),
                                                  'prev_text'),
                                                 ('text_next'...
                                                   'span_len_words',
                                                   'is_first_discourse',
                                                   'is_last_discourse']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['position_bucket'])])),
                ('clf',
                 LogisticRegression(C=1.5,
                                    class_weight={'argument': 0.95,
                                                  'evidence': 1.0,
                                                  'structure': 1.6},
                                    max_iter=5000, n_jobs=-1, random_state=42,
                                    solver='saga'))])